In [19]:
import numpy as np
import matplotlib.pyplot as plt
import emcee
from astropy.io import fits
from halo_model_WL import *
import os
from scipy.linalg import block_diag

In [20]:
hdudes = fits.open('2pt_NG_final_2ptunblind_02_24_21_wnz_redmagic_covupdate.fits')

data_source = hdudes['nz_source'].data[:-1]
data_lens = hdudes['nz_lens'].data[:-1]

header_source = hdudes['nz_source'].header
header_lens = hdudes['nz_lens'].header

#redshift bins Redmagic
z_lims = np.array([0.15,0.35,0.5,0.65,0.8,0.9])

bin_number_source = 4
zs = np.array(data_source['Z_MID'])
zs_bins = []
zs_means = []
nz_source_dict = {}
ngal_source_dict = {}
sige_source_dict = {}
for i in range(bin_number_source):
    z = zs[(zs >= z_lims[i]) & (zs < z_lims[i+1])]
    zs_bins.append(np.array(z))
    zs_means.append(np.mean(z))
    nz_source_dict[i] = data_source['BIN' + str(i+1)]
    ngal_source_dict[i] = header_source['NGAL_' + str(i+1)] * (60 * 180 / np.pi)**2 # gal/arcmin^2  -> gal/str
    sige_source_dict[i] = header_source['SIG_E_' + str(i+1)]


bin_number_lens = 5
zl = np.array(data_lens['Z_MID'])
zl_bins = []
zl_means = []
nz_lens_dict = {}
ngal_lens_dict = {}
for i in range(bin_number_lens):
    z = zl[(zl >= z_lims[i]) & (zl < z_lims[i+1])]
    zl_bins.append(np.array(z))
    zl_means.append(np.mean(z))
    nz_lens_dict[i] = data_lens['BIN' + str(i+1)]
    ngal_lens_dict[i] = header_lens['NGAL_' + str(i+1)] * (60 * 180 / np.pi)**2 # gal/arcmin^2  -> gal/str

#Biases REDMAGIC
galaxy_bias = np.array([1.7, 1.7, 1.7, 2.0, 2.0])  # lens galaxy bias for each lens bin
magnification_bias = np.array([1.3134, -0.5179, 0.3372, 2.2515, 1.9667])  # magnification bias for each lens bin
shear_calibration_bias = np.array([-0.0063,-0.0198,-0.0241,-0.0369])  # shear calibration bias for each source bin

#redshift shift for lens galaxsssies REDMAGIC
shift_params = np.array([0.006,0.001,0.004,-0.002,-0.007])
stretch_params = np.array([1,1,1,1,1.23])

#Intrinsic Alignment (IA)
IA_params = np.array([0.7,-1.36,-1.7,-2.5,1.0,0.62]) #a1,a2,alf1,alfa2,bTA,z0
C1_mean=5e-14 # h^2 Msun^-1 Mpc^3

In [21]:
NSIDE       = 1024
VALID_PAIRS = [(i, j) for i in range(1, 6) for j in range(1, 5) if j > i]

ells_binned = np.loadtxt("ells_binned.csv", delimiter=",")

cov_jk  = np.load('cov_comb_nside1024.npy', allow_pickle=True).item()
cov_dict = {eval(k): v for k, v in cov_jk.items()}
cov      = block_diag(*[cov_dict[(i, j)] for (i, j) in VALID_PAIRS])
cov_inv  = np.linalg.inv(cov)
yerr     = np.sqrt(np.diag(cov))

N_ELL = len(ells_binned)
data_pcls = np.loadtxt(f'pcls_ge_nside{NSIDE}.csv', delimiter=',')

pcls_dict = {(i, j): np.zeros((2, N_ELL)) for (i, j) in VALID_PAIRS}
for row in data_pcls:
    i, j, s = int(row[0]), int(row[1]), int(row[2])
    if (i, j) in pcls_dict:
        pcls_dict[(i, j)][s, :] = row[3:]

# Joint data vector over VALID_PAIRS (E-mode only)
signal = np.concatenate([pcls_dict[(i, j)][0, :] for (i, j) in VALID_PAIRS])


In [22]:
aps = angular_power_spectrum(model = 'subhalomodel',file_dir='pars_v6.npz',DM_type='CDM',tweaks=False)

kmin = 1.00e-04 1/Mpc, kmax = 1.00e+03 1/Mpc
zmin = 0.00, zmax = 2.00
Mmin = 1.00e+01 Msun, Mmax = 1.00e+16 Msun
min subhalo mass = 0.00e+00 Msun
max subhalo mass = 2.46e+15 Msun
Note: redshifts have been re-sorted (earliest first)
Note: redshifts have been re-sorted (earliest first)
Computed Pmm using halo model with subhalos and without tweaks


In [23]:
# data = signal
# inv = np.linalg.inv(yerr)
# #yerr = cov_comb
# sign, logdet_inv = np.linalg.slogdet(inv)
# if sign <= 0:
#     raise ValueError("Inverse covariance matrix is not positive definite.")
#     # det(C) = 1 / det(inv)  =>  log det(C) = -log det(inv)
# logdet_C = -logdet_inv
# N = len(signal)
# log_norm = logdet_C + N * np.log(2 * np.pi)


sign, logdet = np.linalg.slogdet(cov)
if sign <= 0:
    raise ValueError("Covariance matrix is not positive definite.")
N = len(signal)
log_norm = -0.5 * (logdet + N * np.log(2 * np.pi))


In [24]:
from sashimi_c import halo_model
hm = halo_model()
nM     = aps.nM
M_grid = aps.M_grid
z_grid = aps.z_ps
k_grid = aps.k
nz     = len(z_grid)
nk     = len(k_grid)
rhom   = aps.rhom

sigma8_z  = np.zeros(nz)
nu_z      = np.zeros((nz, nM))
f_nu_z    = np.zeros((nz, nM))
n_eff_z   = np.zeros(nz)
cvir_zm   = np.zeros((nz, nM))
rs_zm     = np.zeros((nz, nM))
cvir_sm_zm = np.zeros((nz, nM))
rs_sm_zm  = np.zeros((nz, nM))

for i, z in enumerate(z_grid):
    Plin = aps.Plin[i, :]
    sigma8_z[i] = aps.sigma8_z(Plin, k_grid)

    R      = aps.lagrangian_radius(aps.Mvir_grid[:, i])
    dc     = hm.deltac_func(z) * hm.growthD(z)
    sigmaM = aps.sigma_R(R, Plin, k_grid)
    nu     = dc / sigmaM

    nu_z[i, :]   = nu
    f_nu_z[i, :] = aps.f_sheth_tormen(nu)

    R_nl        = aps.get_nonlinear_radius(R.min(), R.max(), dc, Plin, k_grid)
    n_eff_z[i]  = aps.get_effective_index(R_nl, R, sigmaM)

    Om = aps.Om_at_z(z)
    Dv = hm.Delc(Om - 1) / Om

    for M_id, M in enumerate(M_grid):
        c200  = hm.conc200(M, z)
        r200  = np.cbrt(3 * M / (4 * np.pi * aps.rhocrit0 * hm.g(z) * 200.)) * (1 + z)
        rvir  = R[M_id] / np.cbrt(Dv)
        cvir  = c200 * rvir / r200
        cvir_zm[i, M_id] = cvir
        rs_zm[i, M_id]   = rvir / cvir        

    Msm_grid = aps.Msm_grid[:, i]
    for M_id, M in enumerate(Msm_grid):
        c200    = hm.conc200(M, z)
        r200    = np.cbrt(3 * M / (4 * np.pi * aps.rhocrit0 * hm.g(z) * 200.)) * (1 + z)
        rvir    = R[M_id] / np.cbrt(Dv)
        cvir_sm = c200 * rvir / r200
        cvir_sm_zm[i, M_id] = cvir_sm
        rs_sm_zm[i, M_id]   = rvir / cvir_sm   

P_2h_base  = aps.get_P_dewiggled(z_grid, k_grid).copy() 
one_sup_z  = np.zeros((nz, nk))

for i in range(nz):
    sigma8 = sigma8_z[i]
    k_d    = 0.03841 * sigma8**(-1.089)
    f      = 0.2696  * sigma8**(0.9403)
    P_2h_base[i, :] *= 1 - f * (k_grid / k_d)**2.853 / (1 + (k_grid / k_d)**2.853)

    k_star = 0.03786 * sigma8**(-1.013)
    one_sup_z[i, :] = (k_grid / k_star)**4 / (1 + (k_grid / k_star)**4)


def get_P_fast(theta):
    eta_amp, eta_exp, alpha_amp, alpha_exp = theta

    P_1h    = np.zeros((nz, nk))
    P_total = np.zeros((nz, nk))

    for i, z in enumerate(z_grid):
        sigma8 = sigma8_z[i]
        eta    = eta_amp * sigma8**eta_exp

        nu   = nu_z[i, :]   
        f_nu = f_nu_z[i, :] 
        Mvir = aps.Mvir_grid[:, i] 
        Msm  = aps.Msm_grid[:, i]    
        u = np.zeros((nM, nk))
        for M_id in range(nM):
            nu_eta     = nu[M_id]**eta
            u[M_id, :] = aps.u_nfw(k_grid * nu_eta, cvir_zm[i, M_id], rs_zm[i, M_id])[:, 0]

        usm = np.zeros((nM, nk))
        for M_id in range(nM):
            nu_eta      = nu[M_id]**eta
            usm[M_id, :] = aps.u_nfw(k_grid * nu_eta, cvir_sm_zm[i, M_id], rs_sm_zm[i, M_id])[:, 0]

        w          = Mvir / rhom * f_nu       

        integrand_ss   = w[:, None] * (Msm / Mvir)[:, None]**2 * usm**2
        integrand_sc   = w[:, None] * (Msm / Mvir)[:, None]    * usm * u * aps.I_kmz[:, :, i].T
        integrand_cc   = w[:, None] * u**2 * aps.I_kmz[:, :, i].T**2
        integrand_self = w[:, None] * aps.J_kmz[:, :, i].T    

        P_1h_sm_sm = trapezoid(integrand_ss,   nu, axis=0)
        P_1h_sm_sh = 2 * trapezoid(integrand_sc,   nu, axis=0)
        P_1h_2sh   = trapezoid(integrand_cc,   nu, axis=0)
        P_1h_1sh   = trapezoid(integrand_self, nu, axis=0)

        P_1h[i, :] = (P_1h_sm_sm + P_1h_sm_sh + P_1h_2sh + P_1h_1sh) * one_sup_z[i, :]

        alpha          = alpha_amp * alpha_exp**(n_eff_z[i])
        P_total[i, :]  = (P_2h_base[i, :]**alpha + P_1h[i, :]**alpha)**(1.0 / alpha)

    return P_total 

/Users/user/Desktop/master_project/python_model/halo_model_mcmc/halo_model_WL.py:191: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  sigmaV_squared, _ = quad(lambda k: sigmaV_integrand(k, R, Pk), kmin, kmax, epsabs=0., epsrel=eps)


In [25]:
chi = aps.chi
z   = aps.z

nchi_lens_ij = np.zeros((5, 4, len(chi)))
q_source_ij  = np.zeros((5, 4, len(chi)))

for (i, j) in VALID_PAIRS:   # 1-based
    nz_lens   = nz_lens_dict[i-1]
    nz_source = nz_source_dict[j-1]
    zl_mean   = zl_means[i-1]
    stretch   = stretch_params[i-1]  if stretch_params  is not None else 1
    shift     = shift_params[i-1]    if shift_params    is not None else 0
    shear     = shear_calibration_bias[j-1] if shear_calibration_bias is not None else 0

    nchi_lens_interp   = aps.nz_to_nchi_interp(nz_lens,   zl, zl_mean,
                                                 shift=shift, stretch=stretch)
    nchi_source_interp = aps.nz_to_nchi_interp(nz_source, zs)

    nchi_lens_ij[i-1, j-1, :] = nchi_lens_interp(chi)
    q_source_ij [i-1, j-1, :] = aps.lensing_efficiency(nchi_interp=nchi_source_interp,
                                                         shear=shear)

def get_cls(theta):
    Pmm = get_P_fast(theta)

    cls = np.zeros((5, 4, len(ells_binned)))

    for (i, j) in VALID_PAIRS:   # 1-based
        gb  = galaxy_bias[i-1]
        Pgm = Pmm * gb
        logPgm_interp = aps.make_P_interp(Pgm, log=True)

        nchi_lens = nchi_lens_ij[i-1, j-1, :]
        q_src     = q_source_ij [i-1, j-1, :]

        for m, l in enumerate(ells_binned):
            k   = (l + 0.5) / chi
            pts = np.column_stack([z, k])
            Pgm_chi       = np.exp(logPgm_interp(pts))
            cls[i-1,j-1,m] = simpson(q_src * nchi_lens * Pgm_chi / chi**2, chi)

    # Return joint vector in the same order as VALID_PAIRS
    return np.concatenate([cls[i-1, j-1, :] for (i, j) in VALID_PAIRS])


In [26]:
def lnlike(theta, y):
    m = get_cls(theta)  
    resid = y - m   
    chi2  = np.linalg.multi_dot([resid, cov_inv, resid])
    return -0.5 * (chi2 + log_norm)

In [27]:
def lnprior(theta):
    eta_amp, eta_exp, alpha_amp, alpha_exp = theta
    if 0.0 < eta_amp < 2.0 and -2.0 < eta_exp < 2.0 and 0.0 < alpha_amp < 5.0 and 0.0 < alpha_exp < 5.0:
        return 0.0
    return -np.inf

In [28]:
def lnprob(theta, y):
    lp = lnprior(theta)
    if not np.isfinite(lp):
        return -np.inf

    ll = lnlike(theta, y)
    if not np.isfinite(ll):
        return -np.inf

    return lp + ll

In [29]:
initial = np.array([1.0, 0.0, 2.5, 2.5])  # eta_amp, eta_exp, alpha_amp, alpha_exp
niter = 10000
nwalkers = 64
ndim = len(initial)
p0 = [np.array(initial) + 1.e-4*np.random.randn(ndim) for i in range(nwalkers)]

In [30]:
from multiprocessing import Pool,cpu_count
data=signal
def main(p0, nwalkers, niter, ndim, lnprob, data):
    with Pool() as pool:
        sampler = emcee.EnsembleSampler(
            nwalkers, ndim, lnprob,
            args=(data,),
            pool=pool
        )
        print("Running burn-in...")
        ncpu = cpu_count()
        print("{0} CPUs".format(ncpu))
        p0, _, _ = sampler.run_mcmc(p0, 1000, progress=True)
        sampler.reset()

        print("Running production...")
        ncpu = cpu_count()
        print("{0} CPUs".format(ncpu))
        pos, prob, state = sampler.run_mcmc(p0, niter, progress=True)

    return sampler, pos, prob, state


In [31]:
sampler, pos, prob, state = main(p0,nwalkers,niter,ndim,lnprob,data)

Running burn-in...
4 CPUs


Process SpawnPoolWorker-22:
Process SpawnPoolWorker-21:
Process SpawnPoolWorker-24:
Process SpawnPoolWorker-23:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/anaconda3/envs/camb/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/anaconda3/envs/camb/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/camb/lib/python3.13/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/opt/anaconda3/envs/camb/lib/python3.13/multiprocessing/queues.py", line 387, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
AttributeError: Can't get attribute 'lnprob' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
  File "/opt/anaconda3/envs/camb/lib/

KeyboardInterrupt: 

In [ ]:
samples = sampler.flatchain
smax  = samples[np.argmax(sampler.flatlnprobability)]
print(smax)

NameError: name 'sampler' is not defined

In [ ]:
np.save('samples.npy',samples)